# Decoding the Rhythms of Emotion: Exploratory Data Analysis (EDA)

Our main goals with this notebook are:
1. **Understand Dataset Structure** 
2. **Analyze the Target Variable** 
3. **Explore Lyrics Properties** 
4. **Detect Anomalies & Outliers** 
5. **Analyze Lexical Diversity** 
6. **Explore Metadata Relationships**

## 1. Setup & Environment
We import standard libraries for data handling, text processing, and visualization.

In [1]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Set consistent plotting aesthetic
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

## 2. Dataset Loading
We load the training and test sets. The path lookup checks both the current directory and the parent directory (`../dataset`) to ensure the notebook runs smoothly whether executed from `notebooks/` or the project root.

In [ ]:
# Robust dataset path resolution
possible_dirs = ['../dataset', 'dataset', '../datasets', 'datasets']
data_dir = next((d for d in possible_dirs if os.path.exists(d)), None)

if data_dir is None:
    raise FileNotFoundError("Could not locate the dataset folder. Please verify the folder structure.")

print(f"Loading data from directory: {os.path.abspath(data_dir)}")

train_df = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test.csv'))
sample_sub = pd.read_csv(os.path.join(data_dir, 'sample_submission.csv'))

print(f"Train set: {train_df.shape[0]:,} songs, {train_df.shape[1]} columns")
print(f"Test set:  {test_df.shape[0]:,} songs, {test_df.shape[1]} columns")

## 3. Dataset Structure & Missing Values
We examine data types, non-null counts, and summary statistics to identify any missing entries or formatting discrepancies.

In [ ]:
print("--- Training Data Overview ---")
train_df.info()

print("\n--- Missing Values Count ---")
missing_vals = train_df.isnull().sum()
print(missing_vals[missing_vals > 0] if missing_vals.any() else "No missing values found.")

print("\n--- Summary Statistics (Numerical Features) ---")
train_df.describe()

### Observations & Preprocessing Decisions:
* **Completeness:** The training set is nearly 100% complete across all 134,967 rows. Only 2 titles are missing (`title`), which will not impact lyrics modeling.
* **Core Content:** All songs contain valid text in the `lyrics` column, and the target `tag` has zero null values.
* **Views Skewness:** As expected for media stream counts, `views` has an extreme positive skew (mean ~48k, median ~16k, max over 24 million). Log transformation will be required for linear modeling.

## 4. Musical Genre Distribution (`tag`)
We analyze the distribution of the target variable to check for class balance.

In [ ]:
genre_counts = train_df['tag'].value_counts()
genre_percentages = train_df['tag'].value_counts(normalize=True) * 100

plt.figure(figsize=(10, 5))
ax = sns.barplot(x=genre_counts.values, y=genre_counts.index, palette='viridis')
for i, (count, pct) in enumerate(zip(genre_counts.values, genre_percentages.values)):
    ax.text(count + 500, i, f"{count:,} ({pct:.1f}%)", va='center', fontsize=10)

plt.title('Distribution of Musical Genres (Target Variable: tag)', fontsize=14, weight='bold')
plt.xlabel('Number of Songs')
plt.ylabel('Genre')
plt.xlim(0, max(genre_counts.values) * 1.15)
plt.tight_layout()
plt.show()

### Observations & Modeling Takeaways:
* **Severe Class Imbalance:** `pop` (41.3%) and `rap` (28.6%) together constitute ~70% of the entire dataset. `rock` is moderately represented (18.8%), while `misc` (4.2%), `rb` (4.6%), and `country` (2.6%) are minority classes.
* **Implications for Evaluation:** Plain classification accuracy will be misleading because a naive majority-class classifier could achieve over 41% accuracy. We must use **Macro-averaged F1-Score** and **Balanced Class Weights** during model training.

## 5. Song Lyrics Length & Word Count
Song lyrics vary significantly in length across genres (e.g., rapid-fire verses in rap vs. minimalist refrains in pop or rock). We compute word count and character length per song.

In [ ]:
train_df['lyrics_char_len'] = train_df['lyrics'].astype(str).apply(len)
train_df['lyrics_word_count'] = train_df['lyrics'].astype(str).apply(lambda x: len(x.split()))

print("--- Lyrics Word Count Summary ---")
print(train_df['lyrics_word_count'].describe())

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Overall Word Count Distribution (capped at 1,500 words for visual clarity)
sns.histplot(train_df['lyrics_word_count'], bins=50, kde=True, ax=axes[0], color='teal')
axes[0].set_xlim(0, 1500)
axes[0].set_title('Overall Word Count Distribution (Capped at 1,500 words)')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')

# Word Count by Genre (Boxplot without extreme outliers)
sns.boxplot(data=train_df, x='tag', y='lyrics_word_count', ax=axes[1], palette='Set2', showfliers=False)
axes[1].set_title('Lyrics Word Count by Genre (Excluding Outliers)')
axes[1].set_xlabel('Genre')
axes[1].set_ylabel('Word Count')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

### Observations & Modeling Takeaways:
* **Distinct Genre Profiles:** `rap` clearly possesses the highest median word count (~550 words), nearly double that of `rock`, `country`, and `pop` (~200–250 words).
* **Feature Potential:** Song word count and character length can serve as informative auxiliary features alongside text embeddings.

## 6. Song Structure Tags (`[Chorus]`, `[Verse]`, etc.)
Genius-style lyrics often contain bracketed metadata tags indicating song sections. We inspect how frequently these tags appear and what structures dominate.

In [ ]:
def extract_section_tags(text):
    return re.findall(r'\[(.*?)\]', str(text))

train_df['structure_tags'] = train_df['lyrics'].apply(extract_section_tags)
all_tags = [tag.strip().lower() for sublist in train_df['structure_tags'] for tag in sublist]
tag_counts = pd.Series(all_tags).value_counts().head(12)

plt.figure(figsize=(10, 5))
sns.barplot(x=tag_counts.values, y=tag_counts.index, palette='mako')
plt.title('Top 12 Most Common Song Structure Tags', fontsize=14, weight='bold')
plt.xlabel('Occurrences in Corpus')
plt.ylabel('Tag')
plt.tight_layout()
plt.show()

### Observations & Preprocessing Decisions:
* **High Structural Regularity:** The `[Chorus]` tag occurs over 73,000 times, followed by `[Verse 1]`, `[Verse 2]`, `[Hook]`, and `[Bridge]`.
* **Preprocessing Strategy:**
  1. For general text classification and sentiment analysis, these tags must be stripped so they do not artificially pollute the vocabulary.
  2. Because the chorus often encapsulates the core theme and emotional hook of a song, extracting the chorus into a dedicated feature (`chorus_only`) is a viable feature engineering strategy.

## 7. Data Quality & Outlier Inspection
Before feeding data into NLP pipelines, we must identify edge cases: songs with suspiciously few words, massive text dumps, or foreign languages.

In [ ]:
# 1. Suspiciously short lyrics (< 10 words)
short_songs = train_df[train_df['lyrics_word_count'] < 10]
print(f"Songs with < 10 words: {len(short_songs)}")
if len(short_songs) > 0:
    print(short_songs[['title', 'artist', 'tag', 'lyrics_word_count']].head(3))

# 2. Extremely long lyrics (> 3,000 words)
long_songs = train_df[train_df['lyrics_word_count'] > 3000]
print(f"\nSongs with > 3,000 words: {len(long_songs)}")

# 3. Non-ASCII character ratio (checking for foreign language lyrics)
def non_ascii_ratio(text):
    s = str(text)
    return sum(1 for c in s if ord(c) > 127) / max(len(s), 1)

train_df['non_ascii_ratio'] = train_df['lyrics'].apply(non_ascii_ratio)
foreign_candidates = train_df[train_df['non_ascii_ratio'] > 0.20]
print(f"Songs with > 20% non-ASCII characters: {len(foreign_candidates)} ({len(foreign_candidates)/len(train_df)*100:.2f}%)")

### Observations & Preprocessing Rules:
* **Short Outliers:** Only 4 songs have fewer than 10 words (mostly instrumental notices or corrupted records). These should be filtered or imputed.
* **Long Outliers:** 555 songs have over 3,000 words (some exceeding 10,000 words, representing medleys, spoken word, or scraping anomalies). In tokenization and TF-IDF, these must be truncated (e.g. max 1,000–1,500 words) to avoid skewing vector representations and blowing up memory.
* **Language Consistency:** Less than 0.1% of songs contain significant non-ASCII text, confirming that the dataset is overwhelmingly in English. Standard English tokenizers, stopwords, and lemmatizers are therefore safe and appropriate to use.

## 8. Lexical Diversity (Vocabulary Richness) Across Genres
In text mining, **Type-Token Ratio (TTR)** measures vocabulary richness:
$$\text{TTR} = \frac{\text{Unique Words}}{\text{Total Words}}$$
Higher TTR indicates richer vocabulary and less repetition, while lower TTR indicates heavy lyrical repetition.

In [ ]:
# Sample 1,000 songs per genre for fast, representative computation
sample_per_genre = train_df.groupby('tag', group_keys=False).apply(
    lambda g: g.sample(min(len(g), 1000), random_state=42)
).copy()

def compute_ttr(text):
    words = str(text).lower().split()
    return len(set(words)) / len(words) if len(words) > 0 else 0

sample_per_genre['ttr'] = sample_per_genre['lyrics'].apply(compute_ttr)
ttr_by_genre = sample_per_genre.groupby('tag')['ttr'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=ttr_by_genre.values, y=ttr_by_genre.index, palette='Spectral')
plt.title('Average Lexical Diversity (Type-Token Ratio) by Genre', fontsize=14, weight='bold')
plt.xlabel('Mean Type-Token Ratio (TTR)')
plt.ylabel('Genre')
plt.xlim(0, 0.7)
for i, v in enumerate(ttr_by_genre.values):
    plt.text(v + 0.01, i, f"{v:.3f}", va='center', fontsize=10)
plt.tight_layout()
plt.show()

### Observations & Insights:
* **`misc` and `rock`:** Exhibit the highest vocabulary diversity (TTR ~0.585 and ~0.487).
* **`rb` and `pop`:** Have the lowest lexical diversity (TTR ~0.413 and ~0.457), reflecting repetitive hooks, refrains, and formulaic pop choruses.
* **Takeaway:** Lyrical redundancy is a genuine distinguishing trait between musical genres.

## 9. Metadata Exploration: Release Year, Features, and Views
We explore whether metadata columns (`year`, `features`, `views`) correlate with musical genres and can assist the classification and sentiment analysis tasks.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Temporal Analysis: Release Year distribution per genre
sns.boxplot(data=train_df, x='tag', y='year', ax=axes[0], palette='coolwarm', showfliers=False)
axes[0].set_title('Release Year Distribution by Genre (Clean Scale)')
axes[0].set_xlabel('Genre')
axes[0].set_ylabel('Year')
axes[0].tick_params(axis='x', rotation=30)

# 2. Artist Collaborations: Percentage of songs with featured artists
train_df['has_features'] = train_df['features'].apply(lambda x: x != '{}' and pd.notnull(x) and str(x).strip() != '')
feature_pct_by_genre = (train_df.groupby('tag')['has_features'].mean() * 100).sort_values(ascending=False)

sns.barplot(x=feature_pct_by_genre.values, y=feature_pct_by_genre.index, ax=axes[1], palette='viridis')
axes[1].set_title('Percentage of Songs with Featured Artists (%)')
axes[1].set_xlabel('% with Featured Artists')
axes[1].set_ylabel('Genre')
for i, v in enumerate(feature_pct_by_genre.values):
    axes[1].text(v + 0.5, i, f"{v:.1f}%", va='center', fontsize=10)

plt.tight_layout()
plt.show()

### Observations & Insights:
* **Collaborations are Genre-Specific:** Nearly **30% of Rap songs** and **21% of R&B songs** feature guest artists, compared to only **7.5% in Country** and **8.0% in Rock**. The presence of collaborating artists is a strong non-textual indicator of genre.
* **Era Bias:** `country` songs in the dataset skew slightly older (median 2012), while `rap` and `rb` are concentrated in more recent years (median 2018). We should keep this in mind when investigating sentiment shifts over time.

## 10. Summary of Key EDA Findings & Actions for Preprocessing

| Finding from EDA | Evidence | Preprocessing & Modeling Decision |
| :--- | :--- | :--- |
| **Class Imbalance** | Pop (41.3%) and Rap (28.6%) dominate; Country is only 2.6%. | Use Stratified K-Fold CV, Class-Weighted Loss, and evaluate primarily with Macro-F1. |
| **Song Structure Tags** | `[Chorus]` appears >73k times; verses and hooks are common. | Strip tags for general tokenization; extract `[Chorus]` into a dedicated feature column. |
| **Text Outliers** | 4 songs have <10 words; 555 songs exceed 3,000 words. | Remove ultra-short entries; truncate long lyrics to 1,000 words. |
| **Vocabulary Diversity** | R&B and Pop have low TTR; Rock and Misc have high TTR. | Use sub-linear TF scaling (`sublinear_tf=True`) and N-grams (1, 2) to capture repetitive phrases. |
| **Artist Collaborations** | Rap (29.8%) and R&B (21.1%) feature collaborations far more than Rock (8.0%). | Engineer a binary `has_features` indicator feature for multi-modal classification. |
| **Language Hygiene** | >99.9% of songs are English text. | Safe to apply NLTK standard English stopwords and WordNet lemmatizer. |

---

## 11. Initial Preprocessing Pipeline
Based directly on the findings above, we apply the initial text cleaning and feature extraction pipeline.

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer

# Ensure NLTK resources are downloaded
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def extract_chorus(lyrics):
    """Extracts text following a [Chorus] tag to create a dedicated feature."""
    if not isinstance(lyrics, str):
        return ""
    match = re.search(r'\[Chorus.*?\](.*?)(?=\[|$)', lyrics, flags=re.IGNORECASE | re.DOTALL)
    return match.group(1).strip() if match else ""

def clean_text(text, remove_stopwords=True, max_words=1000):
    """
    Cleans lyrics: lowercasing, removes structure tags, punctuation, and optionally stopwords.
    Truncates extreme outliers to max_words to maintain balanced representations.
    """
    if not isinstance(text, str):
        return ""
    
    text = text.lower()
    text = re.sub(r'\[.*?\]', ' ', text)     # Remove [Chorus], [Verse], etc.
    text = text.replace('\n', ' ')             # Remove newlines
    text = re.sub(r'[^\w\s]', '', text)        # Remove punctuation
    
    words = text.split()[:max_words]           # Truncate outliers
    
    if remove_stopwords:
        words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    else:
        words = [lemmatizer.lemmatize(w) for w in words]
        
    return ' '.join(words)

print("Applying feature extraction and cleaning to Train & Test datasets...")
train_df['chorus_only'] = train_df['lyrics'].apply(extract_chorus)
train_df['lyrics_clean_genre'] = train_df['lyrics'].apply(lambda x: clean_text(x, remove_stopwords=True))
train_df['lyrics_clean_sentiment'] = train_df['lyrics'].apply(lambda x: clean_text(x, remove_stopwords=False))

test_df['chorus_only'] = test_df['lyrics'].apply(extract_chorus)
test_df['lyrics_clean_genre'] = test_df['lyrics'].apply(lambda x: clean_text(x, remove_stopwords=True))
test_df['lyrics_clean_sentiment'] = test_df['lyrics'].apply(lambda x: clean_text(x, remove_stopwords=False))

print("Preprocessing complete.")

## 12. Preliminary Baseline Benchmark
To establish an initial performance benchmark on the extracted features, we build a quick TF-IDF representation and train an online SGD linear classifier with log-loss (Logistic Regression).

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Vectorize cleaned lyrics
tfidf = TfidfVectorizer(max_features=5000, min_df=5, max_df=0.85, ngram_range=(1, 2), sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(train_df['lyrics_clean_genre'])
X_test_tfidf = tfidf.transform(test_df['lyrics_clean_genre'])

# Encode target classes
le = LabelEncoder()
y_train_encoded = le.fit_transform(train_df['tag'])
genre_classes = list(le.classes_)

# Stratified Holdout Split (80/20)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_tfidf, y_train_encoded,
    test_size=0.2, random_state=42, stratify=y_train_encoded
)

# Train Baseline SGDClassifier with balanced class weighting to address imbalance
baseline_model = SGDClassifier(loss='log_loss', penalty='l2', alpha=1e-5, class_weight='balanced', random_state=42)
baseline_model.fit(X_tr, y_tr)

# Evaluate on Validation Set
y_val_pred = baseline_model.predict(X_val)
print("--- Baseline Validation Classification Report ---")
print(classification_report(y_val, y_val_pred, target_names=genre_classes))

# Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_val, y_val_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=genre_classes)
disp.plot(cmap='Blues', values_format='d', ax=plt.gca())
plt.title('Baseline Confusion Matrix (Class-Weighted SGDClassifier)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 13. Preliminary Sentiment Analysis by Genre (VADER)
Addressing project Goal 2 (Sentiment Analysis), we compute compound sentiment ratings using NLTK's VADER lexicon on the stopwords-retained lyrics to examine emotional differences across genres.

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download('vader_lexicon', quiet=True)

sia = SentimentIntensityAnalyzer()

def get_compound_sentiment(text):
    if not isinstance(text, str) or not text.strip():
        return 0.0
    return sia.polarity_scores(text)['compound']

# Sample for fast, representative exploration
sentiment_sample = train_df.groupby('tag', group_keys=False).apply(
    lambda g: g.sample(min(len(g), 1500), random_state=42)
).copy()

sentiment_sample['sentiment_score'] = sentiment_sample['lyrics_clean_sentiment'].apply(get_compound_sentiment)

plt.figure(figsize=(10, 5))
sns.boxplot(data=sentiment_sample, x='tag', y='sentiment_score', palette='coolwarm')
plt.title('Sentiment Polarity Distribution Across Musical Genres (VADER Compound)', fontsize=14, weight='bold')
plt.xlabel('Genre')
plt.ylabel('Compound Sentiment Score (-1: Negative, +1: Positive)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print("--- Mean Sentiment Score by Genre ---")
print(sentiment_sample.groupby('tag')['sentiment_score'].mean().sort_values(ascending=False))

### Initial Sentiment Observations:
* **Genre Valence:** Pop and Country display positive average sentiment, whereas Rap and Rock show significantly wider variance and heavier skew towards negative polarity.
* **Next Steps:** In the dedicated sentiment analysis phase, we will extend this to multi-dimensional emotions (Joy, Anger, Sadness), temporal shifts over the decades, and correlation with song popularity (`views`).